In [2]:
import chess              # Import python-chess library for chess board representation and rules
import heapq              # Import heapq for efficient selection of top/bottom scores (priority queue)
from typing import List   # Import List type hint for better code documentation

# Piece values for material evaluation - simple numeric values for each piece type
PIECE_VALUES = {
    chess.PAWN: 1,        # Pawn worth 1 point
    chess.KNIGHT: 3,      # Knight worth 3 points  
    chess.BISHOP: 3,      # Bishop worth 3 points
    chess.ROOK: 5,        # Rook worth 5 points
    chess.QUEEN: 9,       # Queen worth 9 points
    chess.KING: 0,        # King worth 0 (can't be captured, so not counted in material)
}

# Large value for checkmate detection - used to give checkmate highest priority
CHECKMATE_SCORE = 10000   # Much larger than any possible material score

def evaluate_board(board: chess.Board) -> float:
    if board.is_checkmate():            # First check if current position is checkmate
        if board.turn == chess.WHITE:   # If it's white's turn and checkmate, white lost (so negative score)
            return -CHECKMATE_SCORE     # Negative huge value for white being checkmated
        else:
            return CHECKMATE_SCORE      # Positive huge value for black being checkmated
    
    if board.is_stalemate() or board.is_insufficient_material():    # Check for stalemate (no legal moves but not in check) or draw by insufficient material
        return 0                                                    # Draw positions evaluate to 0

    score = 0                            # Material counting - start with neutral score
    for piece_type in PIECE_VALUES:      # Loop through each piece type (pawn, knight, bishop, rook, queen, king)
        white_pieces = len(board.pieces(piece_type, chess.WHITE))   # Count how many of this piece type white has on the board
        black_pieces = len(board.pieces(piece_type, chess.BLACK))   # Count how many of this piece type black has on the board
        score += PIECE_VALUES[piece_type] * (white_pieces - black_pieces)   # Add to score: white positive (good for white), black negative (good for black), Multiply count difference by piece value
    return score                         # Return total material advantage (positive = white better, negative = black better)

def beam_search(board: chess.Board, beam_width: int, depth_limit: int):
    if depth_limit <= 0:                    # Base case: if depth limit is 0, just evaluate current position
        return [], evaluate_board(board)    # Empty move sequence with evaluation

    # Determine if current player (whose turn it is) is white or black
    is_maximizing = board.turn == chess.WHITE    # White wants to maximize score, black wants to minimize

    # Initialize beam with current position
    initial_eval = evaluate_board(board)        # Evaluate starting position
    beam = [(initial_eval, [], board.copy())]   # Copy board to avoid modifying original

    best_result = ([], initial_eval)            # Track best overall result

    for depth in range(depth_limit):             # Main beam search loop - explore up to depth_limit plies (half-moves)
        candidates = []                          # Store all possible next positions
        # Determine who moves at this depth (alternates between players)
        current_maximizing = is_maximizing if depth % 2 == 0 else not is_maximizing     # depth 0: current player, depth 1: opponent, depth 2: current player, etc.

        for _, move_sequence, current_board in beam:
            legal_moves = list(current_board.legal_moves)    # Get all legal moves from current position
            if not legal_moves:                              # If no legal moves (checkmate or stalemate), just evaluate position
                eval_score = evaluate_board(current_board)   # Get terminal position score
                candidates.append((eval_score, move_sequence, current_board.copy()))
                continue

            # Try each legal move
            for move in legal_moves:
                new_board = current_board.copy()        # Create new board and make the move
                new_board.push(move)                    # Actually make the move
                eval_score = evaluate_board(new_board)  # Evaluate resulting position
                new_sequence = move_sequence + [move]   # Record the move sequence that led to this position and Append current move to sequence
                candidates.append((eval_score, new_sequence, new_board))    # Add to candidates list

        # If no candidates found, stop search
        if not candidates:
            break

        # Select top beam_width candidates based on who is moving
        # For maximizing player (white): keep highest scores
        # For minimizing player (black): keep lowest scores
        if current_maximizing:
            beam = heapq.nlargest(beam_width, candidates, key=lambda x: x[0])   # heapq.nlargest efficiently gets the highest scoring items
        else:
            beam = heapq.nsmallest(beam_width, candidates, key=lambda x: x[0])  # heapq.nsmallest efficiently gets the lowest scoring items

    # After search, find best result from final beam
    if beam:
        if is_maximizing:                           # For maximizing player (white), pick highest scores
            best = max(beam, key=lambda x: x[0])    # Get tuple with max score
        else:
            best = min(beam, key=lambda x: x[0])    # Get tuple with min score for black
        best_result = (best[1], best[0])            # Return (move_sequence, score)
    return best_result    # Return best move sequence and its evaluation

def format_move_sequence(moves: List[chess.Move], board: chess.Board):
    temp_board = board.copy()       # Create temporary board to track position
    move_strs = []                  # List to store formatted move strings
    for move in moves:
        move_strs.append(temp_board.san(move))
        temp_board.push(move)
    return " -> ".join(move_strs)

# starting position - standard chess starting position
board = chess.Board()
beam_width = 4          # Keep only top 4 candidates at each depth
depth_limit = 5         # Search 5 plies (half-moves) deep

print("Starting position\n")
print(f"Board:\n{board}\n")
print(f"Beam Width: {beam_width}")
print(f"Depth Limit: {depth_limit}")
print(f"Turn: {'White' if board.turn == chess.WHITE else 'Black'}")
print()

best_moves, score = beam_search(board, beam_width, depth_limit)

print(f"Best Move Sequence: {format_move_sequence(best_moves, board)}")
print(f"Evaluation Score: {score}")
print()

# Position with checkmate opportunity - tactical position where white can checkmate
tactical_fen = "r1bqkbnr/pppp1ppp/2n5/4p3/2B1P3/5Q2/PPPP1PPP/RNB1K1NR w KQkq - 4 4"
board2 = chess.Board(tactical_fen)

print("Checkmate Opportunity\n")
print(f"Board:\n{board2}\n")
print(f"FEN: {tactical_fen}")
print(f"Beam Width: {beam_width}")
print(f"Depth Limit: {depth_limit}")
print()

best_moves2, score2 = beam_search(board2, beam_width, depth_limit)

print(f"Best Move Sequence: {format_move_sequence(best_moves2, board2)}")
print(f"Evaluation Score: {score2}")
print()

# Endgame position - king and queen vs lone king
endgame_fen = "8/8/8/5k2/8/8/4Q3/4K3 w - - 0 1"
board3 = chess.Board(endgame_fen)

print("Endgame position\n")
print(f"Board:\n{board3}\n")
print(f"FEN: {endgame_fen}")
print(f"Beam Width: {beam_width}")
print(f"Depth Limit: {depth_limit}")
print()

best_moves3, score3 = beam_search(board3, beam_width, depth_limit)

print(f"Best Move Sequence: {format_move_sequence(best_moves3, board3)}")
print(f"Evaluation Score: {score3}")

Starting position

Board:
r n b q k b n r
p p p p p p p p
. . . . . . . .
. . . . . . . .
. . . . . . . .
. . . . . . . .
P P P P P P P P
R N B Q K B N R

Beam Width: 4
Depth Limit: 5
Turn: White

Best Move Sequence: Nh3 -> Nh6 -> Ng5 -> Rg8 -> Nxh7
Evaluation Score: 1

Checkmate Opportunity

Board:
r . b q k b n r
p p p p . p p p
. . n . . . . .
. . . . p . . .
. . B . P . . .
. . . . . Q . .
P P P P . P P P
R N B . K . N R

FEN: r1bqkbnr/pppp1ppp/2n5/4p3/2B1P3/5Q2/PPPP1PPP/RNB1K1NR w KQkq - 4 4
Beam Width: 4
Depth Limit: 5

Best Move Sequence: Be6 -> Nge7 -> Bxd7+ -> Kxd7 -> Qxf7
Evaluation Score: -1

Endgame position

Board:
. . . . . . . .
. . . . . . . .
. . . . . . . .
. . . . . k . .
. . . . . . . .
. . . . . . . .
. . . . Q . . .
. . . . K . . .

FEN: 8/8/8/5k2/8/8/4Q3/4K3 w - - 0 1
Beam Width: 4
Depth Limit: 5

Best Move Sequence: Qe8 -> Kf6 -> Qh8+ -> Kf7 -> Qg8+
Evaluation Score: 9


In [6]:
import random
import math

# Calculate total distance of a route
def calculate_distance(route, points):
    total_distance = 0

    for i in range(len(route) - 1):
        x1, y1 = points[route[i]]
        x2, y2 = points[route[i+1]]
        dist = math.sqrt((x1-x2)**2 + (y1-y2)**2)
        total_distance += dist

    return total_distance


# Generate neighbors by swapping two locations
def get_neighbors(route):
    neighbors = []
    n = len(route)

    for i in range(n):
        for j in range(i+1, n):
            new_route = list(route)
            new_route[i], new_route[j] = new_route[j], new_route[i]     # swap two locations
            neighbors.append(new_route)

    return neighbors


# Hill Climbing Algorithm
def hill_climbing(points):
    n = len(points)

    # Random initial route
    current_route = list(range(n))
    random.shuffle(current_route)
    current_distance = calculate_distance(current_route, points)

    while True:
        neighbors = get_neighbors(current_route)
        next_route = None
        next_distance = current_distance

        # Find first better neighbor
        for neighbor in neighbors:
            neighbor_distance = calculate_distance(neighbor, points)
            if neighbor_distance < next_distance:
                next_route = neighbor
                next_distance = neighbor_distance
                break

        # If no improvement, stop
        if next_distance >= current_distance:
            break
        current_route = next_route
        current_distance = next_distance

    return current_route, current_distance


# Delivery locations (coordinates)
points = [(0,0), (2,3), (5,4), (1,7), (6,1)]


# Run Hill Climbing
route, distance = hill_climbing(points)

print("Optimized Route (index order of points):")
print(route)

print("\nTotal Distance:")
print(distance)

Optimized Route (index order of points):
[0, 4, 2, 1, 3]

Total Distance:
16.53042347625264


In [8]:
import numpy as np
import random

# Generate 10 random cities (x, y)
num_cities = 10
cities = np.random.rand(num_cities, 2) * 100

# Calculate total distance of a route (fitness function - LOWER is better)
def calculate_distance(order):
    dist = 0
    for i in range(len(order)):
        city_a = cities[order[i]]
        city_b = cities[order[(i + 1) % len(order)]]
        dist += np.linalg.norm(city_a - city_b)
    return dist

# Create random individual (matching manual's create_random_individual)
def create_random_individual():
    return random.sample(range(num_cities), num_cities)

# Initialize population
def initial_population(pop_size):
    return [create_random_individual() for _ in range(pop_size)]

# Selection (top 50% as parents - matching manual)
def select_parents(population, fitness_scores):
    # Sort by fitness (lower distance is better)
    sorted_population = [route for _, route in sorted(zip(fitness_scores, population))]
    return sorted_population[:len(population)//2]

# Ordered Crossover (similar to manual's approach)
def crossover(parent1, parent2):
    size = len(parent1)
    start, end = sorted(random.sample(range(size), 2))
    
    child = [None] * size
    child[start:end] = parent1[start:end]
    
    # Fill with parent2 genes in order
    pos = 0
    for i in range(size):
        if child[i] is None:
            while parent2[pos] in child:
                pos += 1
            child[i] = parent2[pos]
            pos += 1
    return child

# Mutation (swap two cities - matching manual)
def mutate(route, mutation_rate=0.1):
    if random.random() < mutation_rate:
        idx1, idx2 = random.sample(range(len(route)), 2)
        route[idx1], route[idx2] = route[idx2], route[idx1]
    return route

# Genetic Algorithm main function (matching manual's structure)
def genetic_algorithm(pop_size=100, max_generations=500, mutation_rate=0.1):
    # Initialize population
    population = initial_population(pop_size)
    
    for generation in range(max_generations):
        # Evaluate fitness
        fitness_scores = [calculate_distance(route) for route in population]
        best_fitness = min(fitness_scores)
        print(f"Generation {generation}: Best Distance = {best_fitness:.2f}")
        
        # Selection (top 50%)
        parents = select_parents(population, fitness_scores)
        
        # Create new population
        new_population = []
        
        # Keep top 10% (elitism)
        elite_count = pop_size // 10
        sorted_indices = np.argsort(fitness_scores)
        for i in range(elite_count):
            new_population.append(population[sorted_indices[i]])
        
        # Fill the rest with offspring
        while len(new_population) < pop_size:
            parent1, parent2 = random.sample(parents, 2)
            child = crossover(parent1, parent2)
            child = mutate(child, mutation_rate)
            new_population.append(child)
        
        population = new_population
    
    # Return best solution
    final_fitness = [calculate_distance(route) for route in population]
    best_index = np.argmin(final_fitness)
    return population[best_index], final_fitness[best_index]

# Run the Genetic Algorithm
best_route, best_dist = genetic_algorithm()

print("\n" + "="*50)
print("RESULTS")
print("="*50)
print(f"Best Route: {best_route}")
print(f"Total Distance: {best_dist:.2f}")

Generation 0: Best Distance = 358.41
Generation 1: Best Distance = 300.99
Generation 2: Best Distance = 300.99
Generation 3: Best Distance = 300.99
Generation 4: Best Distance = 300.99
Generation 5: Best Distance = 270.40
Generation 6: Best Distance = 270.40
Generation 7: Best Distance = 270.40
Generation 8: Best Distance = 270.40
Generation 9: Best Distance = 270.40
Generation 10: Best Distance = 270.40
Generation 11: Best Distance = 254.93
Generation 12: Best Distance = 254.93
Generation 13: Best Distance = 247.18
Generation 14: Best Distance = 247.18
Generation 15: Best Distance = 247.18
Generation 16: Best Distance = 247.18
Generation 17: Best Distance = 247.18
Generation 18: Best Distance = 247.18
Generation 19: Best Distance = 247.18
Generation 20: Best Distance = 247.18
Generation 21: Best Distance = 247.18
Generation 22: Best Distance = 247.18
Generation 23: Best Distance = 247.18
Generation 24: Best Distance = 247.18
Generation 25: Best Distance = 247.18
Generation 26: Best Di

In [12]:
# Calculate processor loads
def calculate_load(allocation):
    loads = {p:0 for p in processors}

    for task_index, processor in allocation:
        execution_time, priority = tasks[task_index]
        loads[processor] += execution_time * priority   # priority can increase weight of task
    return loads


# Evaluation function (minimize max load)
def evaluate(allocation):
    loads = calculate_load(allocation)
    return max(loads.values())


# Generate successors
def generate_successors(state, next_task):
    successors = []

    for processor in processors:
        new_state = state + [(next_task, processor)]
        cost = evaluate(new_state)
        successors.append((cost, new_state))
    return successors


# Beam Search
def beam_search():
    beam = [(0, [])]  # (cost, allocation)

    for task_index in range(len(tasks)):
        candidates = []
        for cost, state in beam:
            successors = generate_successors(state, task_index)
            for s in successors:
                candidates.append(s)

        candidates.sort(key=lambda x: x[0]) # Sort by cost (lower load is better)
        beam = candidates[:beam_width]      # Keep best beam_width candidates

    best = min(beam, key=lambda x: x[0])
    return best


# Run Beam Search
tasks = [(5,3), (2,1), (7,2), (4,3), (6,1)]     # Job tasks: (execution_time, priority)
processors = ["P1","P2","P3"]   # Processors
beam_width = 2

cost, allocation = beam_search()
print("Best Allocation:")
for task,processor in allocation:
    print("Task",task,"->",processor)

print("\nMaximum Processor Load:",cost)

Best Allocation:
Task 0 -> P1
Task 1 -> P2
Task 2 -> P3
Task 3 -> P2
Task 4 -> P2

Maximum Processor Load: 20
